# Training Notebook
This notebook shows how to import code from the `src/` package and run a training loop.

In [ ]:

import sys, os
# Make the generated package importable:
sys.path.append('/home/sagemaker-user/congestion-bogota')
sys.path.append('/home/sagemaker-user/congestion-bogota/data')  # so you can `import sparse_dataloader` if it lives there

import torch
from torch.utils.data import DataLoader
from src.data.collate import pad_collate
from src.models.gnn_temporal import GNN_Temporal_Model
from src.training.loop import train_one_epoch
from src.utils.pickers import build_node_50_picker, EpochUniquePicker

# Your dataset (ensure this module is importable; update the path if needed)
from src.sparse_dataloader import TrafficDataset 

In [ ]:

# ==== Paths (adjust to your repo layout) ====
processed_data_dir = "../data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
static_data_filename = f"{sparse_data_dir}/static_dataset.h5"
dynamic_data_filenames = [
    f"{sparse_data_dir}/coi_points_dataset.h5",
    f"{sparse_data_dir}/coi_tracts_dataset.h5",
    f"{sparse_data_dir}/irregularities_dataset.h5",
    f"{sparse_data_dir}/events_dataset.h5",
    f"{sparse_data_dir}/incidents_dataset.h5",
]
metadata_filenames = [
    f"{sparse_data_dir}/coi_points_dataset_metadata.pkl",
    f"{sparse_data_dir}/coi_tracts_dataset_metadata.pkl",
    f"{sparse_data_dir}/irregularities_dataset_metadata.pkl",
    f"{sparse_data_dir}/events_dataset_metadata.pkl",
    f"{sparse_data_dir}/incidents_dataset_metadata.pkl",
]
target_data_filename = f"{sparse_data_dir}/jams_dataset.h5"
in_timesteps = 150   # adapt to your pipeline
out_timesteps = 50   # adapt to your pipeline
node_batch_size = 16 # how many time windows per seed per batch


In [ ]:

# ==== Build dataset & dataloader ====
traffic_dataset = TrafficDataset(
    graph_path=graph_filename,
    static_data_path=static_data_filename,
    dynamic_data_paths=dynamic_data_filenames,
    metadata_paths=metadata_filenames, 
    target_data_path=target_data_filename,
    in_timesteps=in_timesteps,
    out_timesteps=out_timesteps,
    future_features=[
        "coi_puntos",
        "coi_tramo",
        "AGLOMERACION",
        "EMERGENCIA",
        "MANIFESTACION",
    ],
    node_batch_size=1024,
)

# Standard single-worker loader (seed is updated after each step inside the loop)
loader = DataLoader(traffic_dataset, batch_size=node_batch_size, shuffle=True, num_workers=0, collate_fn=pad_collate)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:

# ==== Build model ====
# Inspect one batch to read dims
batch = next(iter(loader))
edge_index = batch['edge_index']
static_dim   = batch['x_static'].shape[-1]
dynamic_dim  = batch['x_dynamic_past'].shape[-1]
future_dim   = batch['x_dynamic_future'].shape[-1]
input_seq_len    = batch['x_dynamic_past'].shape[2]
forecast_horizon = batch['x_dynamic_future'].shape[2]

model = GNN_Temporal_Model(
    edge_index=edge_index.to(device),
    static_dim=static_dim,
    dynamic_dim=dynamic_dim,
    future_dim=future_dim,
    input_seq_len=input_seq_len,
    forecast_horizon=forecast_horizon,
    model_type="GConvGRU",
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()


In [ ]:

# ==== Build picker (node sampling policy) ====
csv_nodes = f'{processed_data_dir}/nodes_traffic_info.csv'  # update if needed
base_picker = build_node_50_picker(csv_nodes, top_frac=0.2, seed=0)
picker = EpochUniquePicker(base_picker)  # avoid picking the same node twice in an epoch
picker.start_epoch()


In [ ]:

# ==== Train one epoch (timed logs printed) ====
stats = train_one_epoch(
    model=model,
    dataloader=loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    picker=picker,
    traffic_dataset=traffic_dataset,
    epoch_idx=0,
)
print("Epoch stats:", {k: (v if k=='avg_loss' else f"{v*1000:.1f} ms") for k,v in stats.items()})
